# SimpleSequentialChain 基础


任务太大，输出容易飘

可以使用链式拆解

有点像plan模式，只是这回是我人主动拆的了

In [9]:
from langchain.chains import LLMChain, SimpleSequentialChain
from langchain import PromptTemplate
from langchain_openai import ChatOpenAI

LM_STUDIO_BASE = "http://127.0.0.1:1234/v1"
llm = ChatOpenAI(
    base_url=LM_STUDIO_BASE,
    api_key="lm-studio",
    temperature=0,
    model="local",
    request_timeout=120,
    max_tokens=1024,
)

In [8]:
# 任务目标：比较「链式调用」与「单次 Prompt」在输出稳定性上的差异

# Step 1: 根据城市推荐代表美食（仅输出名称）
cate_template = """
你是一位美食博主。
根据城市：{city}，推荐一种最有代表性的当地美食。
要求：只输出美食名称，不要解释，不要标点。
"""
cate_prompt = PromptTemplate(template=cate_template, input_variables=["city"])
cate_chain = LLMChain(llm=llm, prompt=cate_prompt)

# Step 2: 基于美食名称写一句简介（仅一句）
intro_template = """
你是一位美食博主。
根据美食：{cate}，写一句简短介绍，突出特点和口感。
要求：只输出1句话，20-35字，不要分点。
"""
intro_prompt = PromptTemplate(template=intro_template, input_variables=["cate"])
intro_chain = LLMChain(llm=llm, prompt=intro_prompt)

# Step 3: 改写为种草文案（只输出文案）
rewrite_template = """
你是一位小红书美食博主。
将下面这句介绍改写成有食欲、自然、像朋友分享的种草文案：
{intro}
要求：
1) 2-3句话；
2) 60-100字；
3) 只输出文案正文，不要标题，不要标签。
"""
rewrite_prompt = PromptTemplate(template=rewrite_template, input_variables=["intro"])
rewrite_chain = LLMChain(llm=llm, prompt=rewrite_prompt)

# Step 4: 基于文案生成 3 个标题（严格三行）
title_template = """
请基于以下文案生成3个吸引人的中文标题：
{content}
要求：
1) 每个标题10-18字；
2) 不要使用emoji；
3) 严格按以下格式输出三行：
标题1：...
标题2：...
标题3：...
"""
title_prompt = PromptTemplate(template=title_template, input_variables=["content"])
title_chain = LLMChain(llm=llm, prompt=title_prompt)

# 1) 链式调用：分步骤逐步完成任务
sequential_chain = SimpleSequentialChain(
    chains=[cate_chain, intro_chain, rewrite_chain, title_chain],
    verbose=True,
)
chain_result = sequential_chain.run("成都")

# 2) 单次 Prompt：一次性交付完整任务
one_shot_template = """
你是一位小红书美食博主，请一次性完成以下任务：
1) 根据城市 {city} 推荐一种最有代表性的当地美食；
2) 用1句话介绍该美食，突出特点和口感（20-35字）；
3) 改写成种草文案（2-3句话，60-100字，语气自然有食欲）；
4) 基于文案生成3个中文标题（每个10-18字，不要emoji）。

严格按以下格式输出：
美食：...
简介：...
文案：...
标题1：...
标题2：...
标题3：...
"""
one_shot_prompt = PromptTemplate(template=one_shot_template, input_variables=["city"])
one_shot_chain = LLMChain(llm=llm, prompt=one_shot_prompt)
one_shot_result = one_shot_chain.run("成都")

print("\n===== 链式调用结果 =====\n")
print(chain_result)
print("\n===== 单次 Prompt 结果 =====\n")
print(one_shot_result)




> Entering new SimpleSequentialChain chain...
火锅
热汤翻滚，麻辣鲜香，食材涮煮入味，一口下去，暖胃又过瘾。
刚涮完的毛肚嫩到爆，汤底辣得灵魂跳舞，一勺下去暖到心坎里！朋友都说这锅汤能扛住寒风，我直接连喝三碗不腻，配着锅底的香辣，连胃都忍不住打call～
标题1：毛肚嫩到爆辣汤暖胃打call
标题2：一勺汤辣到灵魂起舞
标题3：三碗不腻暖寒风胃动情

> Finished chain.

===== 链式调用结果 =====

标题1：毛肚嫩到爆辣汤暖胃打call
标题2：一勺汤辣到灵魂起舞
标题3：三碗不腻暖寒风胃动情

===== 单次 Prompt 结果 =====

美食：成都担担面  
简介：麻辣鲜香酱料浓郁，面条劲道弹牙，肉末与花生碎碰撞出地道川味。  
文案：一碗担担面，麻辣鲜香在舌尖炸开，面条弹牙入魂，川味灵魂直接拉满！  
标题1：成都必吃担担面，麻辣鲜香太上头  
标题2：这碗担担面，吃出川味灵魂  
标题3：面条弹牙酱香浓，成都人私藏美味


In [ ]:
# 议论文写作：链式调用 vs 单次 Prompt（对比示例）
# 主题示例：年轻人是否应该趁早试错

from langchain.chains import LLMChain, SimpleSequentialChain
from langchain import PromptTemplate

# -----------------------------
# 1) 链式调用：分步骤完成议论文
# -----------------------------

# Step 1: 审题 + 立场（输出中心论点）
thesis_template = """
你是一位议论文写作助手。
请根据题目：{topic}
完成：
1) 提炼关键词（不超过3个）
2) 分析核心争议（1句话）
3) 给出明确立场（支持/反对）
4) 写出中心论点（1句话）

输出要求：
- 只输出最后一行：中心论点：...
- 不要输出其他解释。
"""
thesis_prompt = PromptTemplate(template=thesis_template, input_variables=["topic"])
thesis_chain = LLMChain(llm=llm, prompt=thesis_prompt)

# Step 2: 基于中心论点生成3个分论点
arguments_template = """
根据中心论点：{thesis}
请生成3个分论点。

要求：
1) 每个分论点都能直接支撑中心论点
2) 三个分论点角度不同，且有递进关系
3) 每个分论点1句话，避免空话套话

严格按以下格式输出：
分论点1：...
分论点2：...
分论点3：...
"""
arguments_prompt = PromptTemplate(template=arguments_template, input_variables=["thesis"])
arguments_chain = LLMChain(llm=llm, prompt=arguments_prompt)

# Step 3: 为分论点补论据素材
materials_template = """
根据以下分论点内容，补充议论文素材：
{arguments}

要求：
1) 每个分论点补1个道理论据 + 1个事实/例子 + 1句简要分析
2) 例子尽量具体，不要虚构夸张
3) 输出精炼，便于后续生成提纲
"""
materials_prompt = PromptTemplate(template=materials_template, input_variables=["arguments"])
materials_chain = LLMChain(llm=llm, prompt=materials_prompt)

# Step 4: 生成提纲
outline_template = """
请根据以下素材生成议论文提纲：
{materials}

要求：
1) 按“开头-分论点一-分论点二-分论点三-结尾”组织
2) 每部分写出要点（不是全文）
3) 结构清晰，逻辑递进

输出标题：
议论文提纲
"""
outline_prompt = PromptTemplate(template=outline_template, input_variables=["materials"])
outline_chain = LLMChain(llm=llm, prompt=outline_prompt)

# Step 5: 按提纲写正文（输出完整短文）
essay_template = """
请根据以下提纲写一篇议论文正文：
{outline}

要求：
1) 600-800字
2) 包含开头、3个主体段、结尾
3) 观点明确，论证具体，衔接自然
4) 语言正式、简洁、有逻辑
"""
essay_prompt = PromptTemplate(template=essay_template, input_variables=["outline"])
essay_chain = LLMChain(llm=llm, prompt=essay_prompt)

# Step 6: 统一润色
polish_template = """
请润色下面这篇议论文：
{essay}

要求：
1) 保持原观点不变
2) 优化段落衔接与逻辑连贯性
3) 删除重复表达，增强书面感
4) 输出最终稿，不要额外说明
"""
polish_prompt = PromptTemplate(template=polish_template, input_variables=["essay"])
polish_chain = LLMChain(llm=llm, prompt=polish_prompt)

essay_sequential_chain = SimpleSequentialChain(
    chains=[thesis_chain, arguments_chain, materials_chain, outline_chain, essay_chain, polish_chain],
    verbose=True,
)

essay_chain_result = essay_sequential_chain.run("年轻人是否应该趁早试错")


# -----------------------------
# 2) 单次 Prompt：一次性完成议论文
# -----------------------------

one_shot_essay_template = """
你是一位议论文写作助手，请围绕题目：{topic}
一次性完成以下任务：
1) 给出中心论点（立场明确）
2) 给出3个分论点（角度不同、逻辑递进）
3) 每个分论点补1个道理论据和1个事实例子
4) 先给出简要提纲
5) 再写一篇600-800字议论文（含开头、3个主体段、结尾）

输出格式：
中心论点：...
分论点1：...
分论点2：...
分论点3：...
提纲：...
正文：...
"""
one_shot_essay_prompt = PromptTemplate(template=one_shot_essay_template, input_variables=["topic"])
one_shot_essay_chain = LLMChain(llm=llm, prompt=one_shot_essay_prompt)
one_shot_essay_result = one_shot_essay_chain.run("年轻人是否应该趁早试错")

print("\n===== 议论文链式调用结果 =====\n")
print(essay_chain_result)
print("\n===== 议论文单次 Prompt 结果 =====\n")
print(one_shot_essay_result)